# Reproducible Data Cleaning

**Prepared by:** Khant Min Zaw (6845028)  
**Role:** Project Lead and Data Curator

This notebook documents the cleaning and validation of the UCI Student Performance Portuguese-course dataset. The tested functions in `scripts/clean_data.py` remain the single source of truth, so the notebook explains and executes the pipeline without duplicating its cleaning rules.

Valid unusual observations, including high absence values and zero final grades, are retained rather than automatically removed or capped.

## 1. Setup and portable project paths

The project root is discovered from the current working directory. This allows the notebook to run on Windows, macOS, or Linux when Jupyter is started anywhere inside the repository.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start=None):
    """Locate the repository without using a personal absolute path."""
    current = Path(start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        expected_file = candidate / "data" / "raw" / "student_performance.csv"
        if (candidate / "src").is_dir() and expected_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter from inside "
        "the Statistics-superstars repository."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.clean_data import (
    clean_student_data,
    validate_values,
    write_cleaning_log,
)
from src.data_loader import (
    load_processed_data,
    load_raw_data,
    save_processed_data,
)

print("Project root:", PROJECT_ROOT)

## 2. Load the standardized raw dataset

The primary dataset contains records from the Portuguese-language course. Loading it through `src/data_loader.py` validates the expected 33-column schema and column order.

In [ ]:
raw_data = load_raw_data()

print(f"Raw dataset shape: {raw_data.shape[0]} rows x {raw_data.shape[1]} columns")
print(raw_data.head().to_string())

## 3. Initial data-quality assessment

Missing values and exact duplicate rows are checked before cleaning. Data types are also summarized to confirm that numeric and categorical variables were parsed correctly.

In [ ]:
quality_before = pd.DataFrame(
    {
        "Check": [
            "Rows",
            "Columns",
            "Missing values",
            "Exact duplicate rows",
            "Numeric columns",
            "Categorical columns",
        ],
        "Result": [
            raw_data.shape[0],
            raw_data.shape[1],
            int(raw_data.isna().sum().sum()),
            int(raw_data.duplicated().sum()),
            raw_data.select_dtypes(include="number").shape[1],
            raw_data.select_dtypes(exclude="number").shape[1],
        ],
    }
)

print(quality_before.to_string(index=False))

## 4. Execute the cleaning and validation rules

The reusable cleaning function normalizes categorical whitespace, checks missing values and duplicates, validates documented numeric ranges and category values, and records every decision.

In [ ]:
cleaned_data, log_entries = clean_student_data(raw_data)
validate_values(cleaned_data)

print("Cleaning actions:")
for entry in log_entries:
    print("-", entry)

## 5. Review unusual observations

The IQR rule is used only to flag unusual absence values for review. Flagged records are not automatically errors. Zero values in `G3` are also retained because zero is within the documented grade range of 0–20.

In [ ]:
q1 = cleaned_data["absences"].quantile(0.25)
q3 = cleaned_data["absences"].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - (1.5 * iqr)
upper_limit = q3 + (1.5 * iqr)

absence_flags = ~cleaned_data["absences"].between(
    lower_limit,
    upper_limit,
)

unusual_values = pd.DataFrame(
    {
        "Review item": [
            "Absence Q1",
            "Absence Q3",
            "Absence IQR",
            "IQR upper limit",
            "IQR-flagged absence records retained",
            "G3 zero-grade records retained",
        ],
        "Value": [
            q1,
            q3,
            iqr,
            upper_limit,
            int(absence_flags.sum()),
            int(cleaned_data["G3"].eq(0).sum()),
        ],
    }
)

print(unusual_values.to_string(index=False))

## 6. Save reproducible outputs

The validated dataset and deterministic cleaning log are written using the same functions as the command-line pipeline.

In [ ]:
processed_path = save_processed_data(cleaned_data)
log_path = write_cleaning_log(log_entries)
reloaded_data = load_processed_data()

pd.testing.assert_frame_equal(cleaned_data, reloaded_data)

print("Cleaned dataset:", processed_path)
print("Cleaning log:", log_path)
print("Saved shape:", reloaded_data.shape)

## 7. Final verification

These assertions prevent the notebook from silently completing when an expected cleaning result changes.

In [ ]:
verification = {
    "Final shape is 649 x 33": cleaned_data.shape == (649, 33),
    "No missing values": int(cleaned_data.isna().sum().sum()) == 0,
    "No duplicate rows": int(cleaned_data.duplicated().sum()) == 0,
    "All documented values valid": True,
    "Valid observations preserved": len(cleaned_data) == len(raw_data),
    "Raw and cleaned data are equal": raw_data.equals(cleaned_data),
    "Twenty-one absence records retained": int(absence_flags.sum()) == 21,
    "Fifteen G3 zero grades retained": int(cleaned_data["G3"].eq(0).sum()) == 15,
}

verification_table = pd.DataFrame(
    verification.items(),
    columns=["Validation check", "Passed"],
)

print(verification_table.to_string(index=False))

assert verification_table["Passed"].all()
print("All cleaning and validation checks passed.")

## Conclusion

The dataset required no imputation, duplicate removal, type conversion, outlier capping, or row deletion. The cleaned dataset therefore remains identical to the standardized raw dataset. The value of this pipeline is reproducibility: it verifies the schema, documented ranges, categories, unusual observations, and saved outputs every time it runs.

Detailed actions are recorded in `reports/cleaning_log.txt`.